# 🫁 cough-vision — Pulmonary TB Detection from Chest X-rays

> **Kaggle T4×2 DDP Training Notebook** — DenseNet-121 + Conv-stem ViT-S/16 Ensemble  
> WHO TPP target: **≥90% sensitivity at ≥70% specificity**

## Multi-GPU strategy: DistributedDataParallel (DDP)
```
mp.spawn → 2 × ddp_worker (one per T4 GPU, NCCL backend)
         ├─ DistributedWeightedSampler  → unique balanced shard per GPU
         ├─ CutMixMixUpCollator         → per-GPU augmented batches
         ├─ torch.cuda.amp.GradScaler   → fp16 on each GPU
         ├─ gradient accumulation       → effective batch = 16×2×2 = 64
         ├─ dist.all_gather_object      → AUC from ALL validation samples
         ├─ dist.broadcast              → early-stop signal synced across GPUs
         └─ rank-0 only                 → checkpoint save, W&B log
```

## Notebook flow
0. Environment + GPU detection  
1. Install dependencies  
2. Clone / mount source code  
3. Dataset paths & CSV builder (Shenzhen, Montgomery, TBX11K)  
4. Preprocessing sanity check  
5. Hyperparameters  
6. DDP utilities (DistributedWeightedSampler, setup/cleanup)  
7. Model smoke-test (single GPU, forward pass)  
8. DDP training worker function  
9. Launch training (mp.spawn for T4×2)  
10. Training curves  
11. Evaluation — WHO TPP report  
12. Grad-CAM visualisation  
13. Per-site threshold calibration  
14. ONNX export  
15. Save artefacts  


---
## 0 · Environment — GPU detection

In [ ]:
import os, sys, subprocess, pathlib, platform, socket, datetime

# ── Kaggle vs local ───────────────────────────────────────────────────────
IS_KAGGLE = os.path.exists('/kaggle/input')
WORK_DIR  = pathlib.Path('/kaggle/working') if IS_KAGGLE else pathlib.Path('.')
WORK_DIR.mkdir(parents=True, exist_ok=True)

print(f'Platform : {platform.platform()}')
print(f'Python   : {sys.version.split()[0]}')
print(f'Kaggle   : {IS_KAGGLE}')
print(f'Work dir : {WORK_DIR}')

import torch

# ── Multi-GPU inventory ───────────────────────────────────────────────────
WORLD_SIZE = torch.cuda.device_count()
DEVICE     = 'cuda' if WORLD_SIZE > 0 else 'cpu'

print(f'\nGPU count : {WORLD_SIZE}')
for i in range(WORLD_SIZE):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}  : {p.name}  VRAM={p.total_memory/1e9:.1f} GB')

print(f'\nPyTorch   : {torch.__version__}')
print(f'CUDA      : {torch.version.cuda}')

if WORLD_SIZE > 1:
    print(f'\n✓ T4×2 detected — DDP (NCCL) will be used.')
    print(f'  NCCL_P2P_DISABLE=1 set (Kaggle T4 uses PCIe, not NVLink)')
elif WORLD_SIZE == 1:
    print(f'\n→ Single GPU — ddp_worker called directly (no mp.spawn overhead)')
else:
    print(f'\n→ CPU mode — training will be slow, use GPU for real runs')

# ── Global seed ───────────────────────────────────────────────────────────
import random, numpy as np
MASTER_SEED = 42
torch.manual_seed(MASTER_SEED)
random.seed(MASTER_SEED)
np.random.seed(MASTER_SEED)
print(f'\nSeed set to {MASTER_SEED}')


---
## 1 · Install dependencies

In [ ]:
# Kaggle ships: torch, torchvision, numpy, pandas, matplotlib, scikit-learn
# Only extras needed:
EXTRA_PKGS = [
    'timm>=0.9.0',
    'opencv-python-headless>=4.8.0',
    'onnx>=1.14.0',
    'onnxruntime>=1.15.0',
    'grad-cam>=1.4.0',
    'pydicom>=2.4.0',
    'tqdm>=4.65.0',
    'wandb>=0.15.0',
]

import subprocess, sys
for pkg in EXTRA_PKGS:
    ret = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', pkg],
        capture_output=True, text=True,
    )
    print(f'{"✓" if ret.returncode == 0 else "✗"} {pkg}')

print('Done.')


---
## 2 · Clone / mount source code

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# On Kaggle: add this notebook's dataset as input OR clone from GitHub.
# The block below tries both paths gracefully.
# ──────────────────────────────────────────────────────────────────────────

SRC_DIR = None

# Option A — repo uploaded as Kaggle dataset
kaggle_code = pathlib.Path('/kaggle/input/cough-vision/src')
if kaggle_code.exists():
    SRC_DIR = kaggle_code.parent
    print(f'Found code at Kaggle dataset: {SRC_DIR}')

# Option B — clone from GitHub (requires internet; enable in Kaggle settings)
if SRC_DIR is None:
    clone_target = WORK_DIR / 'cough-vision'
    if not clone_target.exists():
        print('Cloning cough-vision from GitHub …')
        ret = subprocess.run(
            ['git', 'clone', '--depth', '1',
             'https://github.com/YOUR_ORG/cough-vision.git',
             str(clone_target)],
            capture_output=True, text=True
        )
        print(ret.stdout or ret.stderr)
    SRC_DIR = clone_target

# Option C — already running from the repo root
if SRC_DIR is None:
    SRC_DIR = pathlib.Path('.')

assert SRC_DIR is not None
SRC_ROOT = SRC_DIR / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))
print(f'Source root on sys.path: {SRC_ROOT}')

# Quick import smoke-test
from config import get_config
cfg = get_config('densenet_vit_ensemble')
print(f'Config loaded — CNN={cfg.cnn.name}, ViT={cfg.vit.name}')

---
## 3 · Dataset paths & CSV builder

### Expected Kaggle input layout
```
/kaggle/input/
├── pulmonary-chest-xray-abnormalities/
│   ├── shenzhen/CXR_png/   (662 PNGs)
│   └── montgomery/CXR_png/ (138 PNGs)
├── tbx11k/
│   ├── imgs/train/ + imgs/val/
│   └── labels.csv
└── chest-xray-pneumonia/   (optional extra negatives)
```


In [ ]:
import pathlib, csv, json, random, math
import numpy as np

# ── Kaggle input roots (adjust if your dataset slugs differ) ───────────────
KAGGLE_INPUT = pathlib.Path('/kaggle/input') if IS_KAGGLE else pathlib.Path('data')

SHENZHEN_IMG  = KAGGLE_INPUT / 'pulmonary-chest-xray-abnormalities' / 'shenzhen'  / 'CXR_png'
SHENZHEN_META = KAGGLE_INPUT / 'pulmonary-chest-xray-abnormalities' / 'shenzhen'  / 'shenzhen_metadata.csv'
MONTGOMERY_IMG  = KAGGLE_INPUT / 'pulmonary-chest-xray-abnormalities' / 'montgomery' / 'CXR_png'
MONTGOMERY_META = KAGGLE_INPUT / 'pulmonary-chest-xray-abnormalities' / 'montgomery' / 'montgomery_metadata.csv'
TBX11K_DIR    = KAGGLE_INPUT / 'tbx11k'

DATA_DIR = WORK_DIR / 'data'
DATA_DIR.mkdir(exist_ok=True)
IMG_SYMLINK = DATA_DIR / 'images'

# ── Dataset availability check ────────────────────────────────────────────
print('Dataset availability:')
for name, path in [
    ('Shenzhen images',   SHENZHEN_IMG),
    ('Montgomery images', MONTGOMERY_IMG),
    ('TBX11K',           TBX11K_DIR),
]:
    status = '✓' if path.exists() else '✗ NOT FOUND'
    print(f'  {name}: {status} ({path})')

In [ ]:
# ── Build unified metadata CSV ────────────────────────────────────────────
# Columns: image_path, tb_label, findings_label, active_inactive_label,
#           site, view_position, split

records = []
SEED    = 42
random.seed(SEED)

# ── Shenzhen ─────────────────────────────────────────────────────────────
if SHENZHEN_IMG.exists():
    for img_path in sorted(SHENZHEN_IMG.glob('*.png')):
        fname = img_path.name
        # Shenzhen naming: CHNCXR_XXXX_0.png = normal, CHNCXR_XXXX_1.png = TB
        tb_label = int(fname.split('_')[-1].replace('.png', ''))
        records.append({
            'image_path':           str(img_path),
            'tb_label':             tb_label,
            'findings_label':       '0,0,0,0,0,0',
            'active_inactive_label': -1,
            'site':                 'shenzhen',
            'view_position':        'PA',
        })
    print(f'Shenzhen: {len([r for r in records if r["site"]=="shenzhen"])} images')

# ── Montgomery ────────────────────────────────────────────────────────────
if MONTGOMERY_IMG.exists():
    before = len(records)
    for img_path in sorted(MONTGOMERY_IMG.glob('*.png')):
        fname    = img_path.name
        tb_label = int(fname.split('_')[-1].replace('.png', ''))
        records.append({
            'image_path':           str(img_path),
            'tb_label':             tb_label,
            'findings_label':       '0,0,0,0,0,0',
            'active_inactive_label': -1,
            'site':                 'montgomery',
            'view_position':        'PA',
        })
    print(f'Montgomery: {len(records)-before} images')

# ── TBX11K (weakly labelled) ──────────────────────────────────────────────
if TBX11K_DIR.exists():
    label_csv = TBX11K_DIR / 'labels.csv'
    before = len(records)
    if label_csv.exists():
        with open(label_csv) as f:
            for row in csv.DictReader(f):
                img_p = TBX11K_DIR / 'imgs' / row.get('file_name', row.get('image_path', ''))
                if not img_p.exists():
                    continue
                raw_label = str(row.get('label', row.get('class', '0')))
                # TBX11K: 'tb' or 'sick_but_non-tb' or 'healthy'
                tb_label = 1 if 'tb' in raw_label.lower() and 'non' not in raw_label.lower() else 0
                records.append({
                    'image_path':           str(img_p),
                    'tb_label':             tb_label,
                    'findings_label':       '0,0,0,0,0,0',
                    'active_inactive_label': -1,
                    'site':                 'tbx11k',
                    'view_position':        'PA',
                })
    print(f'TBX11K: {len(records)-before} images')

print(f'\nTotal records: {len(records)}')
print(f'TB positive  : {sum(r["tb_label"]==1 for r in records)}')
print(f'Normal       : {sum(r["tb_label"]==0 for r in records)}')

In [ ]:
# ── Stratified split (site + label aware) ────────────────────────────────
sys.path.insert(0, str(SRC_ROOT))  # ensure src is importable
from data.dataset import stratified_split

if len(records) == 0:
    # ── DEMO MODE: generate tiny synthetic CSV for notebook testing ────────
    print('⚠  No real images found — running in DEMO MODE with synthetic data')
    demo_dir = DATA_DIR / 'demo_images'
    demo_dir.mkdir(exist_ok=True)
    import cv2
    for i in range(200):
        img = np.random.randint(20, 220, (256, 256), dtype=np.uint8)
        p   = demo_dir / f'img_{i:04d}.png'
        cv2.imwrite(str(p), img)
        records.append({
            'image_path':           str(p),
            'tb_label':             i % 5 == 0,   # 20% TB
            'findings_label':       '0,0,0,0,0,0',
            'active_inactive_label': -1,
            'site':                 'demo',
            'view_position':        'PA',
        })
    print(f'Demo records created: {len(records)}')

train_recs, val_recs, test_recs = stratified_split(
    records, val_fraction=0.10, test_fraction=0.10, seed=SEED
)

def write_csv(path, rows, split_name):
    fieldnames = ['image_path','tb_label','findings_label',
                  'active_inactive_label','site','view_position','split']
    with open(path, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        for r in rows:
            r['split'] = split_name
            w.writerow({k: r.get(k,'') for k in fieldnames})

write_csv(DATA_DIR / 'train.csv', train_recs, 'train')
write_csv(DATA_DIR / 'val.csv',   val_recs,   'val')
write_csv(DATA_DIR / 'test.csv',  test_recs,  'test')

print(f'\nSplit summary:')
print(f'  train : {len(train_recs):>4}  '
      f'(TB={sum(r["tb_label"]==1 for r in train_recs)}, '
      f'Norm={sum(r["tb_label"]==0 for r in train_recs)})')
print(f'  val   : {len(val_recs):>4}  '
      f'(TB={sum(r["tb_label"]==1 for r in val_recs)}, '
      f'Norm={sum(r["tb_label"]==0 for r in val_recs)})')
print(f'  test  : {len(test_recs):>4}  '
      f'(TB={sum(r["tb_label"]==1 for r in test_recs)}, '
      f'Norm={sum(r["tb_label"]==0 for r in test_recs)})')

---
## 4 · Preprocessing sanity check

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import cv2
from data.preprocessing import (
    load_image, apply_clahe, apply_gaussian_denoise,
    preprocess_cxr, passes_qc
)

# Pick a random training image for demo
sample_path = train_recs[0]['image_path']
sample_label = train_recs[0]['tb_label']

raw  = load_image(sample_path)                       # (H, W) uint8
clhe = apply_clahe(raw, clip_limit=2.5)              # CLAHE enhanced
dns  = apply_gaussian_denoise(clhe, sigma=0.5)       # denoised
out  = preprocess_cxr(raw, target_size=224)          # full pipeline → (3, 224, 224)

print(f'Image : {sample_path}')
print(f'Label : {"TB" if sample_label else "Normal"}')
print(f'Raw   : shape={raw.shape}  dtype={raw.dtype}  '
      f'min={raw.min()}  max={raw.max()}')
print(f'QC    : {"PASS" if passes_qc(raw) else "FAIL"}')
print(f'Output: shape={out.shape}  dtype={out.dtype}  '
      f'min={out.min():.3f}  max={out.max():.3f}')

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
titles = ['Raw CXR', 'After CLAHE', 'After Denoise', 'Network input (ch 0)']
imgs   = [raw, clhe, dns, out[0]]
for ax, img, title in zip(axes, imgs, titles):
    ax.imshow(img, cmap='gray', vmin=img.min(), vmax=img.max())
    ax.set_title(title, fontsize=11)
    ax.axis('off')
plt.suptitle(f'Preprocessing pipeline — {"TB" if sample_label else "Normal"} case',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(WORK_DIR / 'preprocessing_check.png', dpi=100)
plt.show()
print('Saved → preprocessing_check.png')

---
## 5 · Hyperparameters

> `batch_size` is **per GPU**. Effective batch = `batch_size × WORLD_SIZE × accum_steps`

| Config | Value | Note |
|---|---|---|
| batch_size | 16 | per GPU |
| WORLD_SIZE | 2 | T4×2 |
| accum_steps | 2 | gradient accumulation |
| **Effective batch** | **64** | 16 × 2 × 2 |


In [ ]:
HP = dict(
    # Model
    preset          = 'densenet_vit_ensemble',
    cnn_backbone    = 'densenet121',
    vit_backbone    = 'vit_small_patch16_384',
    pretrained      = 'imagenet',      # change to 'mocov3_cxr' + supply ckpt for best results

    # Training  (batch_size is PER-GPU)
    max_epochs      = 60,
    freeze_epochs   = 3,               # Phase 2a: heads-only warm-up
    batch_size      = 16,              # per GPU -> effective = 16 x 2 GPUs x 2 accum = 64
    accum_steps     = 2,
    backbone_lr     = 1e-5,
    head_lr         = 1e-3,
    weight_decay    = 1e-4,
    warmup_epochs   = 5,
    patience        = 10,              # early-stop patience

    # Loss
    focal_gamma     = 2.0,
    focal_alpha     = 0.75,
    label_smoothing = 0.1,
    cls_weight      = 1.0,             # alpha
    findings_weight = 0.3,             # beta
    active_weight   = 0.2,             # gamma

    # Augmentation
    use_cutmix      = True,
    use_mixup       = True,
    pos_weight      = 5.0,             # TB positive up-sampling factor

    # WHO evaluation
    who_sensitivity = 0.90,

    # Misc
    mixed_precision = True,
    seed            = 42,
    num_workers     = 2,               # per GPU (Kaggle total-worker limit)
    port            = 12355,           # MASTER_PORT; auto-bumped if occupied
)

eff_bs = HP['batch_size'] * max(1, WORLD_SIZE) * HP['accum_steps']
print(f"Effective batch size = {eff_bs}  "
      f"({HP['batch_size']} per-GPU x {max(1,WORLD_SIZE)} GPUs x {HP['accum_steps']} accum)")

import json
(WORK_DIR / 'hyperparameters.json').write_text(json.dumps(HP, indent=2))
print("Saved -> hyperparameters.json")


---
## 6 · DDP utilities

- **`DistributedWeightedSampler`**: combines class-imbalance weighting with per-rank non-overlapping sharding so every GPU sees a balanced, unique subset.
- **`find_free_port`**: avoids MASTER_PORT collisions if 12355 is in use.
- **`setup_ddp` / `cleanup_ddp`**: NCCL process-group lifecycle with `NCCL_P2P_DISABLE=1` for Kaggle PCIe topology.


In [ ]:
import torch
import torch.distributed as dist
import torch.multiprocessing as mp
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import Sampler
from torch.utils.data.distributed import DistributedSampler
import socket, math, os, datetime


def find_free_port(preferred: int = 12355) -> int:
    """Return preferred port if free, else find any free port."""
    for port in [preferred] + list(range(49152, 49252)):
        try:
            with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
                s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
                s.bind(('', port))
                return port
        except OSError:
            continue
    raise RuntimeError('No free port found in range 49152-49252')


def setup_ddp(rank: int, world_size: int, port: int) -> None:
    """Initialise NCCL process group for one DDP worker."""
    os.environ.update({
        'MASTER_ADDR': 'localhost',
        'MASTER_PORT': str(port),
        # Kaggle T4s use PCIe, not NVLink -- disabling P2P avoids hangs
        'NCCL_P2P_DISABLE': '1',
        'NCCL_IB_DISABLE':  '1',
    })
    dist.init_process_group(
        backend    = 'nccl',
        rank       = rank,
        world_size = world_size,
        timeout    = datetime.timedelta(minutes=60),
    )
    torch.cuda.set_device(rank)


def cleanup_ddp() -> None:
    if dist.is_available() and dist.is_initialized():
        dist.destroy_process_group()


class DistributedWeightedSampler(Sampler):
    """
    Combines WeightedRandomSampler and DistributedSampler.

    Each rank sees a non-overlapping shard of a globally weighted draw so
    class-imbalance correction (TB pos_weight=5) is preserved across all
    GPUs while each GPU gets unique samples.

    Args:
        weights      : per-sample float weights (len == len(dataset)).
        num_replicas : world_size (number of DDP processes).
        rank         : rank of this process.
        replacement  : sample with replacement (recommended for small datasets).
        seed         : base seed; epoch is added via set_epoch() each epoch.
    """

    def __init__(
        self,
        weights: list,
        num_replicas: int,
        rank: int,
        replacement: bool = True,
        seed: int = 42,
    ) -> None:
        self.weights      = torch.as_tensor(weights, dtype=torch.float64)
        self.num_replicas = num_replicas
        self.rank         = rank
        self.replacement  = replacement
        self.seed         = seed
        self.epoch        = 0
        # Samples this rank draws per epoch
        self.num_samples  = math.ceil(len(weights) / num_replicas)
        self.total_size   = self.num_samples * num_replicas

    def set_epoch(self, epoch: int) -> None:
        self.epoch = epoch

    def __iter__(self):
        g = torch.Generator()
        g.manual_seed(self.seed + self.epoch * 10007)
        indices = torch.multinomial(
            self.weights,
            self.total_size,
            replacement=self.replacement,
            generator=g,
        ).tolist()
        # Stride-based shard: rank 0 gets [0, W, 2W, ...], rank 1 gets [1, W+1, ...]
        shard = indices[self.rank : self.total_size : self.num_replicas]
        return iter(shard)

    def __len__(self) -> int:
        return self.num_samples


print("DDP utilities defined.")
print(f"  WORLD_SIZE = {WORLD_SIZE}")
print("  DistributedWeightedSampler, setup_ddp, cleanup_ddp, find_free_port ready.")


---
## 7 · Model smoke-test

Quick single-process forward pass before spawning DDP workers.

In [ ]:
from config import get_config
from models import build_ensemble

cfg = get_config(HP['preset'])
cfg.cnn.name         = HP['cnn_backbone']
cfg.vit.name         = HP['vit_backbone']
cfg.cnn.pretrained   = HP['pretrained']
cfg.vit.pretrained   = HP['pretrained']
cfg.train.output_dir = str(WORK_DIR / 'checkpoints')

_dev = torch.device('cuda:0' if WORLD_SIZE > 0 else 'cpu')
_m   = build_ensemble(cfg).to(_dev)

total_p    = sum(p.numel() for p in _m.parameters())
trainable_p= sum(p.numel() for p in _m.parameters() if p.requires_grad)
print(f"Total params     : {total_p/1e6:.2f} M")
print(f"Trainable params : {trainable_p/1e6:.2f} M")

_m.eval()
with torch.no_grad():
    _o = _m(torch.zeros(2, 3, 224, 224, device=_dev))

print(f"\nOutput keys : {list(_o.keys())}")
print(f"  tb_logits  : {_o['tb_logits'].shape}")
print(f"  tb_prob    : {_o['tb_prob'].tolist()}")
print(f"  findings   : {_o['findings_logits'].shape}")
print("\n✓ Forward pass OK -- cleaning up test instance")

del _m, _o
if _dev.type == 'cuda':
    torch.cuda.empty_cache()


---
## 8 · DDP training worker

`ddp_worker(rank, world_size, args)` is the **complete training pipeline** for one GPU. Launched via `mp.spawn` (T4×2) or called directly (single GPU).

| Step | What happens |
|---|---|
| **Init** | NCCL process group, `set_device(rank)` |
| **Model** | `build_ensemble` → `DDP(model, device_ids=[rank])` |
| **Sampler** | `DistributedWeightedSampler` (balanced + unique per GPU) |
| **Phase 2a** | `model.freeze_backbones()`, heads-only warm-up |
| **Phase 2b** | `model.unfreeze_backbones()`, full fine-tune |
| **Gradient sync** | DDP `all_reduce` every backward pass automatically |
| **AUC** | `dist.all_gather_object` merges probs from all ranks |
| **Early stop** | rank-0 decides → `dist.broadcast` to all ranks |
| **Checkpoint** | rank-0 saves `model.module.state_dict()` |


In [ ]:
def ddp_worker(rank: int, world_size: int, args: dict) -> None:
    """
    Complete DDP training worker -- one per GPU.
    Launched by mp.spawn (rank 0..world_size-1) or called directly.
    All imports happen INSIDE because mp.spawn forks fresh processes.
    """
    import os, sys, math, time, json, datetime, warnings
    import pathlib

    # 0. Process group + device ─────────────────────────────────────────
    sys.path.insert(0, args['src_root'])
    setup_ddp(rank, world_size, args['port'])
    is_main = (rank == 0)
    device  = torch.device(f'cuda:{rank}')
    torch.manual_seed(args['seed'] + rank * 137)   # unique seed per rank

    if is_main:
        eff_bs = args['batch_size'] * world_size * args['accum_steps']
        print(f"[DDP] {world_size} processes | batch/GPU={args['batch_size']} | "
              f"effective_batch={eff_bs}")

    # 1. Imports (must be inside worker for clean pickle) ───────────────
    from config import get_config
    from models import build_ensemble
    from training.losses import MultiTaskLoss
    from training.scheduler import build_optimizer, cosine_schedule_with_warmup
    from data.dataset import TBDataset, compute_sample_weights
    from data.augmentation import (
        get_train_transform, get_inference_transform, CutMixMixUpCollator,
    )
    from torch.utils.data import DataLoader
    from torch.utils.data.distributed import DistributedSampler
    import numpy as np

    _roc_auc = None
    try:
        from sklearn.metrics import roc_auc_score as _roc_auc
    except ImportError:
        pass

    # 2. Config ─────────────────────────────────────────────────────────
    cfg = get_config(args['preset'])
    cfg.cnn.name         = args['cnn_backbone']
    cfg.vit.name         = args['vit_backbone']
    cfg.cnn.pretrained   = args['pretrained']
    cfg.vit.pretrained   = args['pretrained']
    output_dir = pathlib.Path(args['output_dir'])
    output_dir.mkdir(parents=True, exist_ok=True)

    # 3. Model -> DDP ────────────────────────────────────────────────────
    model = build_ensemble(cfg).to(device)
    # DDP.__getattr__ delegates unknown attrs to .module, so
    # model.freeze_backbones() / model.get_parameter_groups() work directly.
    model = DDP(model, device_ids=[rank], output_device=rank,
                find_unused_parameters=False)

    # 4. Datasets ────────────────────────────────────────────────────────
    root = pathlib.Path('/')   # CSV stores absolute paths
    train_ds = TBDataset(
        csv_path   = args['train_csv'],
        image_root = root,
        split      = 'train',
        transform  = get_train_transform(
            image_size         = 224,
            rotation_degrees   = 10.0,
            translate_fraction = 0.05,
            scale_range        = (0.85, 1.15),
            brightness_jitter  = 0.20,
            contrast_jitter    = 0.20,
        ),
    )
    val_ds = TBDataset(
        csv_path   = args['val_csv'],
        image_root = root,
        split      = 'val',
        transform  = get_inference_transform(image_size=224),
    )

    # 5. Samplers ────────────────────────────────────────────────────────
    # Train: weighted + distributed (balanced shard per GPU)
    tb_lbls = [int(r.get('tb_label', 0)) for r in train_ds.records]
    weights = compute_sample_weights(tb_lbls, pos_weight=args['pos_weight'])
    train_sampler = DistributedWeightedSampler(
        weights, num_replicas=world_size, rank=rank, seed=args['seed'],
    )
    # Val: plain distributed, no weighting, no shuffle
    val_sampler = DistributedSampler(
        val_ds, num_replicas=world_size, rank=rank, shuffle=False,
    )

    collator = CutMixMixUpCollator(
        n_classes    = 2,
        cutmix_alpha = 1.0,
        mixup_alpha  = 0.4,
        cutmix_prob  = 0.5 if args['use_cutmix'] else 0.0,
        mixup_prob   = 0.5 if args['use_mixup']  else 0.0,
    )
    nw = args['num_workers']
    train_loader = DataLoader(
        train_ds, sampler=train_sampler, batch_size=args['batch_size'],
        num_workers=nw, pin_memory=True, drop_last=True,
        collate_fn=collator, persistent_workers=(nw > 0),
    )
    val_loader = DataLoader(
        val_ds, sampler=val_sampler, batch_size=args['batch_size'] * 2,
        num_workers=nw, pin_memory=True, persistent_workers=(nw > 0),
    )

    # 6. Loss / optimiser / scheduler / scaler ───────────────────────────
    criterion = MultiTaskLoss(
        cls_weight      = args['cls_weight'],
        findings_weight = args['findings_weight'],
        active_weight   = args['active_weight'],
        focal_gamma     = args['focal_gamma'],
        focal_alpha     = args['focal_alpha'],
        label_smoothing = args['label_smoothing'],
    ).to(device)

    bl_lr   = args['backbone_lr']
    h_lr    = args['head_lr']
    wd      = args['weight_decay']
    accum   = args['accum_steps']
    max_ep  = args['max_epochs']
    frz_ep  = args['freeze_epochs']
    wrm_ep  = args['warmup_epochs']
    pat     = args['patience']
    mp_flag = args['mixed_precision']

    # Phase 2a: freeze backbones
    model.freeze_backbones()   # DDP forwards to model.module
    optimizer = build_optimizer(model, bl_lr, h_lr, wd)

    steps_ep     = len(train_loader)
    total_steps  = max_ep * steps_ep
    warmup_steps = wrm_ep * steps_ep
    scheduler    = cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)
    scaler       = torch.cuda.amp.GradScaler(enabled=mp_flag)

    best_auc   = 0.0
    no_improve = 0
    history    = []

    # 7. Training loop ───────────────────────────────────────────────────
    for epoch in range(max_ep):
        t0 = time.time()
        train_sampler.set_epoch(epoch)

        # Phase 2b: unfreeze
        if epoch == frz_ep:
            model.unfreeze_backbones()
            optimizer = build_optimizer(model, bl_lr, h_lr, wd)
            scheduler = cosine_schedule_with_warmup(optimizer, 0, total_steps)
            if is_main:
                print(f'[Epoch {epoch:03d}] Phase 2b: backbones unfrozen')

        # ── TRAIN ────────────────────────────────────────────────────────
        model.train()
        run_loss = torch.zeros(1, device=device)
        n_steps  = 0
        optimizer.zero_grad()

        for step, batch in enumerate(train_loader):
            if isinstance(batch[1], dict):
                images    = batch[0].to(device, non_blocking=True)
                ld        = batch[1]
                tb_labels = ld['labels_a'].to(device, non_blocking=True)
                lam       = float(ld.get('lam', 1.0))
                labels_b  = ld['labels_b'].to(device, non_blocking=True)
                findings  = torch.zeros(images.shape[0], 6, device=device)
                active    = torch.full((images.shape[0],), -1,
                                       dtype=torch.long, device=device)
            else:
                images, tb_raw, find_raw, act_raw, *_ = batch
                images    = images.to(device, non_blocking=True)
                tb_labels = tb_raw.to(device, non_blocking=True)
                findings  = find_raw.to(device, non_blocking=True)
                active    = act_raw.to(device, non_blocking=True)
                lam, labels_b = 1.0, None

            with torch.cuda.amp.autocast(enabled=mp_flag):
                out = model(images)
                ld_ = criterion(out, tb_labels, findings, active,
                                lam=lam, labels_b=labels_b)
                loss = ld_['loss'] / accum

            scaler.scale(loss).backward()
            run_loss += loss.detach() * accum

            if (step + 1) % accum == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer); scaler.update()
                scheduler.step()
                optimizer.zero_grad()
            n_steps += 1

        # Flush remaining partial batch
        if (len(train_loader) % accum) != 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
            scheduler.step()
            optimizer.zero_grad()

        # Average train loss across ALL GPUs
        dist.all_reduce(run_loss, op=dist.ReduceOp.AVG)
        avg_train = float(run_loss / max(1, n_steps))

        # ── VALIDATE ─────────────────────────────────────────────────────
        model.eval()
        vl_t   = torch.zeros(1, device=device)
        l_prob : list = []
        l_lbl  : list = []

        with torch.no_grad():
            for batch in val_loader:
                if isinstance(batch[1], dict):
                    images    = batch[0].to(device, non_blocking=True)
                    tb_labels = batch[1]['labels_a'].to(device, non_blocking=True)
                    findings  = torch.zeros(images.shape[0], 6, device=device)
                    active    = torch.full((images.shape[0],), -1,
                                          dtype=torch.long, device=device)
                else:
                    images, tb_raw, find_raw, act_raw, *_ = batch
                    images    = images.to(device, non_blocking=True)
                    tb_labels = tb_raw.to(device, non_blocking=True)
                    findings  = find_raw.to(device, non_blocking=True)
                    active    = act_raw.to(device, non_blocking=True)

                out = model(images)
                vl_t += criterion(out, tb_labels, findings, active)['loss'].detach()
                l_prob.extend(out['tb_prob'].cpu().tolist())
                lbs = (tb_labels.argmax(dim=-1) if tb_labels.dim() > 1
                       else tb_labels).cpu().tolist()
                l_lbl.extend(lbs)

        # Average val loss across GPUs
        dist.all_reduce(vl_t, op=dist.ReduceOp.AVG)
        avg_val = float(vl_t / max(1, len(val_loader)))

        # ── GATHER predictions from ALL ranks for global AUC ─────────────
        # Each rank only saw 1/world_size of val_ds. Merge to compute
        # AUC on the full validation set.
        all_p = [None] * world_size
        all_l = [None] * world_size
        dist.all_gather_object(all_p, l_prob)
        dist.all_gather_object(all_l, l_lbl)

        # ── METRICS + CHECKPOINT (rank 0 only) ────────────────────────────
        if is_main:
            flat_p = [v for sub in all_p for v in sub]
            flat_l = [v for sub in all_l for v in sub]
            val_auc = 0.5
            if _roc_auc is not None and len(set(flat_l)) > 1:
                try:
                    val_auc = float(_roc_auc(np.array(flat_l), np.array(flat_p)))
                except Exception:
                    val_auc = 0.5

            elapsed = time.time() - t0
            row = dict(epoch=epoch, train_loss=avg_train,
                       val_loss=avg_val, val_auc=val_auc, elapsed_s=elapsed)
            history.append(row)

            print(f"Epoch {epoch:03d} | "
                  f"train={avg_train:.4f} | "
                  f"val_loss={avg_val:.4f} | "
                  f"val_auc={val_auc:.4f} | "
                  f"{elapsed:.1f}s")

            if val_auc > best_auc:
                best_auc   = val_auc
                no_improve = 0
                try:
                    # model.module = the underlying TBEnsemble (strips DDP wrapper)
                    torch.save({
                        'epoch'              : epoch,
                        'model_state_dict'   : model.module.state_dict(),
                        'optimizer_state_dict': optimizer.state_dict(),
                        'val_auc'            : val_auc,
                        'history'            : history,
                    }, output_dir / 'best_model.pt')
                except Exception as e:
                    warnings.warn(f'Checkpoint save failed: {e}')
            else:
                no_improve += 1

            try:
                torch.save(
                    {'epoch': epoch, 'model_state_dict': model.module.state_dict()},
                    output_dir / 'last_model.pt',
                )
            except Exception:
                pass

            should_stop = int(no_improve >= pat)
        else:
            should_stop = 0

        # Broadcast early-stop decision from rank 0 to all ranks
        stop_t = torch.tensor(should_stop, dtype=torch.int32, device=device)
        dist.broadcast(stop_t, src=0)
        dist.barrier()    # sync before next epoch

        if stop_t.item():
            if is_main:
                print(f"Early stopping at epoch {epoch} "
                      f"(no improvement for {pat} epochs).")
            break

    # 8. Final artefacts (rank 0) + cleanup ─────────────────────────────
    if is_main:
        print(f"\nTraining complete. Best val AUC = {best_auc:.4f}")
        (output_dir / 'history.json').write_text(json.dumps(history, indent=2))

    cleanup_ddp()


print("ddp_worker defined.")


---
## 9 · Launch training

- **T4×2**: `mp.spawn` starts 2 worker processes (`nprocs=WORLD_SIZE`), each pinned to one GPU via `torch.cuda.set_device(rank)`.
- **Single GPU / CPU**: worker is called directly — no inter-process overhead.

> `ARGS` is a plain `dict` (picklable). `wandb_run` objects are **not** picklable and must not be passed into workers — W&B can be initialised inside the worker separately if needed.

In [ ]:
# Optional W&B (rank-0 parent process only for the run URL)
wandb_run = None
try:
    import wandb
    key = os.environ.get('WANDB_API_KEY', '')
    if key:
        wandb.login(key=key, relogin=True)
        wandb_run = wandb.init(
            project = 'cough-vision',
            name    = f'{HP["preset"]}_{HP["max_epochs"]}ep_{WORLD_SIZE}gpu',
            config  = HP,
            dir     = str(WORK_DIR),
        )
        print('W&B run:', wandb_run.url)
    else:
        print('W&B skipped (no WANDB_API_KEY). Add it in Kaggle Secrets -> Environment.')
except ImportError:
    print('wandb not installed -- skipping W&B')

CKPT_DIR = str(WORK_DIR / 'checkpoints')

# Build args dict (every value MUST be picklable for mp.spawn)
ARGS = dict(
    src_root        = str(SRC_ROOT),
    preset          = HP['preset'],
    cnn_backbone    = HP['cnn_backbone'],
    vit_backbone    = HP['vit_backbone'],
    pretrained      = HP['pretrained'],
    max_epochs      = HP['max_epochs'],
    freeze_epochs   = HP['freeze_epochs'],
    batch_size      = HP['batch_size'],
    accum_steps     = HP['accum_steps'],
    backbone_lr     = HP['backbone_lr'],
    head_lr         = HP['head_lr'],
    weight_decay    = HP['weight_decay'],
    warmup_epochs   = HP['warmup_epochs'],
    patience        = HP['patience'],
    focal_gamma     = HP['focal_gamma'],
    focal_alpha     = HP['focal_alpha'],
    label_smoothing = HP['label_smoothing'],
    cls_weight      = HP['cls_weight'],
    findings_weight = HP['findings_weight'],
    active_weight   = HP['active_weight'],
    use_cutmix      = HP['use_cutmix'],
    use_mixup       = HP['use_mixup'],
    pos_weight      = HP['pos_weight'],
    mixed_precision = HP['mixed_precision'],
    seed            = HP['seed'],
    num_workers     = HP['num_workers'],
    train_csv       = str(DATA_DIR / 'train.csv'),
    val_csv         = str(DATA_DIR / 'val.csv'),
    output_dir      = CKPT_DIR,
    port            = find_free_port(HP['port']),
)

print(f"MASTER_PORT  : {ARGS['port']}")
print(f"WORLD_SIZE   : {WORLD_SIZE}")
print(f"Strategy     : {'DDP mp.spawn (T4x2)' if WORLD_SIZE > 1 else 'Single process'}")
print()

if WORLD_SIZE > 1:
    # ── T4×2: fork one worker per GPU ────────────────────────────────────
    # nprocs=WORLD_SIZE spawns WORLD_SIZE copies of ddp_worker in parallel.
    # join=True blocks this cell until ALL workers finish or raise.
    mp.spawn(
        ddp_worker,
        args   = (WORLD_SIZE, ARGS),
        nprocs = WORLD_SIZE,
        join   = True,
    )
else:
    # ── Single GPU or CPU: call directly (no inter-process overhead) ──────
    ddp_worker(rank=0, world_size=1, args=ARGS)

if wandb_run is not None:
    try:
        wandb_run.finish()
    except Exception:
        pass

print("Training complete.")


---
## 10 · Training curves

In [ ]:
import matplotlib.pyplot as plt, json, pathlib

hist_path = pathlib.Path(CKPT_DIR) / 'history.json'
if not hist_path.exists():
    bp = pathlib.Path(CKPT_DIR) / 'best_model.pt'
    history = torch.load(str(bp), map_location='cpu').get('history', []) if bp.exists() else []
else:
    history = json.loads(hist_path.read_text())

if not history:
    print("No history found -- run training first.")
else:
    epochs     = [h['epoch']      for h in history]
    train_loss = [h['train_loss'] for h in history]
    val_loss   = [h['val_loss']   for h in history]
    val_auc    = [h['val_auc']    for h in history]
    best_ep    = max(history, key=lambda h: h['val_auc'])['epoch']
    best_auc   = max(h['val_auc'] for h in history)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(epochs, train_loss, label='Train loss', color='steelblue', lw=2)
    ax1.plot(epochs, val_loss,   label='Val loss',   color='tomato',    lw=2)
    ax1.axvline(HP['freeze_epochs'], color='gray', ls='--', alpha=0.6,
                label=f'Unfreeze (ep {HP["freeze_epochs"]})')
    ax1.set(xlabel='Epoch', ylabel='Loss', title='Loss curves')
    ax1.legend(); ax1.grid(alpha=0.3)

    ax2.plot(epochs, val_auc, color='forestgreen', marker='o', ms=3, lw=2)
    ax2.axhline(0.90, color='red', ls='--', lw=1.5, label='WHO 90% target')
    ax2.axvline(HP['freeze_epochs'], color='gray', ls='--', alpha=0.6)
    ax2.scatter([best_ep], [best_auc], color='gold', s=120, zorder=5,
                label=f'Best AUC={best_auc:.4f} (ep {best_ep})')
    ax2.set(xlabel='Epoch', ylabel='AUC-ROC', title='Validation AUC (global, all GPUs)')
    ax2.legend(); ax2.grid(alpha=0.3)

    plt.suptitle(
        f'cough-vision Training — {WORLD_SIZE} GPU(s), eff. batch={HP["batch_size"]*max(1,WORLD_SIZE)*HP["accum_steps"]}',
        fontsize=13, fontweight='bold',
    )
    plt.tight_layout()
    plt.savefig(WORK_DIR / 'training_curves.png', dpi=120)
    plt.show()
    print(f"Best val AUC = {best_auc:.4f} at epoch {best_ep}")


---
## 11 · Evaluation — WHO TPP report

In [ ]:
import json
import numpy as np
from evaluation.metrics import evaluate, find_who_threshold, auc_metrics
from config import get_config
from models import build_ensemble
from data.dataset import TBDataset
from data.augmentation import get_inference_transform
from torch.utils.data import DataLoader

ckpt_path = pathlib.Path(CKPT_DIR) / 'best_model.pt'
_ecfg = get_config(HP['preset'])
_ecfg.cnn.name = HP['cnn_backbone']; _ecfg.vit.name = HP['vit_backbone']
eval_model = build_ensemble(_ecfg)

if ckpt_path.exists():
    ckpt = torch.load(str(ckpt_path), map_location='cpu')
    eval_model.load_state_dict(ckpt['model_state_dict'])
    print(f"Loaded best model (epoch={ckpt.get('epoch','?')}, val_auc={ckpt.get('val_auc',0):.4f})")
else:
    print("WARNING: no checkpoint found -- evaluating untrained model")

eval_dev   = torch.device('cuda:0' if WORLD_SIZE > 0 else 'cpu')
eval_model = eval_model.to(eval_dev)
eval_model.eval()

test_ds = TBDataset(
    csv_path=DATA_DIR / 'test.csv',
    image_root=pathlib.Path('/'),
    split='test',
    transform=get_inference_transform(224),
)
test_loader = DataLoader(test_ds, batch_size=HP['batch_size']*2,
                         shuffle=False, num_workers=HP['num_workers'],
                         pin_memory=(eval_dev.type=='cuda'))

all_probs: list = []
all_lbls:  list = []
with torch.no_grad():
    for batch in test_loader:
        imgs, lbls, *_ = batch
        out = eval_model(imgs.to(eval_dev))
        all_probs.extend(out['tb_prob'].cpu().tolist())
        all_lbls.extend(lbls.cpu().tolist())

y_true = np.array(all_lbls,  dtype=int)
y_prob = np.array(all_probs, dtype=float)

report = evaluate(y_true, y_prob,
                  sensitivity_target=HP['who_sensitivity'],
                  n_bootstrap=1000, site='test_set')
(WORK_DIR / 'eval_report.json').write_text(json.dumps(report, indent=2, default=str))

print('\n' + '='*60 + '\nEVALUATION REPORT\n' + '='*60)
try:
    print(f"AUC-ROC  : {report['auc']['auc_roc']:.4f}  "
          f"(95% CI [{report['auc_roc_ci']['ci_lower']:.4f}, "
          f"{report['auc_roc_ci']['ci_upper']:.4f}])")
    print(f"AUC-PR   : {report['auc']['auc_pr']:.4f}")
except (KeyError, TypeError): pass
try:
    who  = report['who_operating_point']
    mets = report['metrics_at_who_threshold']
    tpp  = "YES" if who.get('who_tpp_met') else "NO"
    print(f"\nWHO TPP ({HP['who_sensitivity']:.0%} sens / 70% spec): {tpp}")
    for k in ('threshold','sensitivity','specificity'):
        print(f"  {k:<12} : {who[k]:.4f}")
    for k in ('ppv','npv','f1','mcc'):
        print(f"  {k:<12} : {mets.get(k,0):.4f}")
except (KeyError, TypeError): pass
try:
    cal = report['calibration']
    print(f"\n  ECE         : {cal['ece']:.4f}")
    print(f"  Brier score : {cal['brier']:.4f}")
except (KeyError, TypeError): pass
print('='*60)


In [ ]:
# ── ROC + PR curves ────────────────────────────────────────────────────────
from sklearn.metrics import roc_curve, precision_recall_curve, auc

fpr, tpr, _   = roc_curve(y_true, y_prob)
prec, rec, _  = precision_recall_curve(y_true, y_prob)
roc_auc_val   = auc(fpr, tpr)
pr_auc_val    = auc(rec, prec)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# ROC
ax1.plot(fpr, tpr, lw=2, color='steelblue',
         label=f'ROC AUC = {roc_auc_val:.4f}')
ax1.plot([0,1],[0,1], 'k--', alpha=0.4, label='Random')
try:
    who_t = report['who_operating_point']['threshold']
    who_fpr_val = report['who_operating_point'].get('fpr',
                  1.0 - report['who_operating_point']['specificity'])
    who_tpr_val = report['who_operating_point']['sensitivity']
    ax1.scatter([who_fpr_val], [who_tpr_val], s=120, color='red', zorder=5,
                label=f'WHO op. point (t={who_t:.3f})')
except Exception:
    pass
ax1.axhline(0.90, color='red', linestyle=':', alpha=0.6, label='90% sensitivity')
ax1.set_xlabel('False Positive Rate'); ax1.set_ylabel('True Positive Rate')
ax1.set_title('ROC Curve'); ax1.legend(loc='lower right'); ax1.grid(alpha=0.3)

# PR
ax2.plot(rec, prec, lw=2, color='forestgreen',
         label=f'PR AUC = {pr_auc_val:.4f}')
baseline = y_true.mean()
ax2.axhline(baseline, color='gray', linestyle='--',
            label=f'Prevalence = {baseline:.1%}')
ax2.set_xlabel('Recall (Sensitivity)'); ax2.set_ylabel('Precision (PPV)')
ax2.set_title('Precision–Recall Curve'); ax2.legend(); ax2.grid(alpha=0.3)

plt.suptitle('cough-vision Evaluation — Test Set', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(WORK_DIR / 'roc_pr_curves.png', dpi=120)
plt.show()

In [ ]:
# ── Reliability diagram (calibration) ─────────────────────────────────────
from sklearn.calibration import calibration_curve

fig, ax = plt.subplots(figsize=(7, 6))
fraction_pos, mean_pred = calibration_curve(y_true, y_prob, n_bins=10)
ax.plot(mean_pred, fraction_pos, 's-', color='steelblue', label='Model')
ax.plot([0,1],[0,1], 'k--', alpha=0.5, label='Perfect calibration')
ax.set_xlabel('Mean predicted probability')
ax.set_ylabel('Fraction of positives')
ax.set_title('Reliability Diagram (Calibration)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(WORK_DIR / 'calibration_curve.png', dpi=120)
plt.show()

---
## 12 · Grad-CAM visualisation

In [ ]:
import cv2
from models.interpretability import GradCAM, get_target_layer

# Attach Grad-CAM hooks to the full ensemble (CNN last-conv layer → tb_logits)
model.enable_gradcam()

# Pick the first 4 TB-positive test cases
tb_test_recs = [r for r in test_ds.records if int(r.get('tb_label', 0)) == 1][:4]
if not tb_test_recs:
    tb_test_recs = test_ds.records[:4]  # fallback

fig, axes = plt.subplots(len(tb_test_recs), 3,
                          figsize=(12, 4 * len(tb_test_recs)))
if len(tb_test_recs) == 1:
    axes = [axes]

for row_i, rec in enumerate(tb_test_recs):
    img_path = rec['image_path']

    # Load raw image for display
    raw = load_image(img_path)
    H, W = raw.shape

    # Get preprocessed tensor
    import PIL.Image as PilImage
    pil = PilImage.fromarray(apply_clahe(raw)).convert('RGB')
    img_t = val_tf(pil).unsqueeze(0).to(DEVICE)

    # Forward
    model.eval()
    with torch.no_grad():
        out = model(img_t)
    score = float(out['tb_score'].cpu())

    # Grad-CAM
    try:
        heatmap = model._gradcam(img_t, class_idx=1)  # (H', W') float [0,1]
        overlay = model._gradcam.overlay(heatmap, raw)
    except Exception as e:
        print(f'Grad-CAM failed for {img_path}: {e}')
        heatmap = np.zeros((H, W))
        overlay = cv2.cvtColor(raw, cv2.COLOR_GRAY2RGB)

    # Heatmap resized to raw image size
    heat_disp = cv2.resize(heatmap, (W, H))

    axes[row_i][0].imshow(raw, cmap='gray')
    axes[row_i][0].set_title(f'Original  label={rec["tb_label"]}', fontsize=10)
    axes[row_i][0].axis('off')

    im = axes[row_i][1].imshow(heat_disp, cmap='jet', vmin=0, vmax=1)
    axes[row_i][1].set_title(f'Grad-CAM  score={score:.1f}/100', fontsize=10)
    axes[row_i][1].axis('off')
    plt.colorbar(im, ax=axes[row_i][1], fraction=0.046, pad=0.04)

    axes[row_i][2].imshow(overlay)
    axes[row_i][2].set_title('Overlay (α=0.5)', fontsize=10)
    axes[row_i][2].axis('off')

plt.suptitle('Grad-CAM: TB-positive cases', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(WORK_DIR / 'gradcam_tb_cases.png', dpi=120)
plt.show()

model.disable_gradcam()
print('Grad-CAM overlays saved → gradcam_tb_cases.png')

---
## 13 · Per-site calibration

In [ ]:
from evaluation.calibration import build_calibrator, PlattCalibrator
from evaluation.metrics import find_who_threshold

# Use the test set as stand-in for "local site data"
# In production: use 50-200 locally labelled CXRs from the deployment site.

cal = build_calibrator(method='platt')
cal.fit(y_prob, y_true, sensitivity_target=HP['who_sensitivity'])

cal_probs = cal.predict_proba(y_prob)
who_cal   = find_who_threshold(y_true, cal_probs, sensitivity_target=HP['who_sensitivity'])

print(f'Calibrated threshold : {cal.threshold_:.4f}')
print(f'Post-calibration WHO operating point:')
print(f'  Sensitivity : {who_cal["sensitivity"]:.1%}')
print(f'  Specificity : {who_cal["specificity"]:.1%}')
print(f'  WHO TPP met : {"✓ YES" if who_cal["who_tpp_met"] else "✗ NO"}')

cal_save_path = WORK_DIR / 'calibrator.json'
cal.save(cal_save_path)
print(f'Calibrator saved → {cal_save_path}')

---
## 14 · ONNX export

In [ ]:
from deployment.export import export_onnx, optimise_onnx

onnx_path = WORK_DIR / 'cough_vision.onnx'

print('Exporting to ONNX …')
try:
    exported = export_onnx(
        model          = model,
        output_path    = onnx_path,
        cnn_input_size = 224,
        opset          = 17,
        dynamic_axes   = True,
        model_version  = 'v1.0.0',
    )
    size_mb = exported.stat().st_size / 1e6
    print(f'ONNX exported ({size_mb:.1f} MB) → {exported}')

    # Optional graph optimisation
    try:
        opt_path = optimise_onnx(exported)
        print(f'Optimised ONNX → {opt_path}')
    except ImportError:
        print('onnxoptimizer not installed — skipping graph optimisation')

except Exception as e:
    print(f'ONNX export failed: {e}\n(This is expected if torch.onnx cannot trace dynamic ViT ops)')

---
## 15 · Save artefacts

In [ ]:
import shutil

artefacts = {
    'best_model.pt'        : pathlib.Path(cfg.train.output_dir) / 'best_model.pt',
    'last_model.pt'        : pathlib.Path(cfg.train.output_dir) / 'last_model.pt',
    'calibrator.json'      : WORK_DIR / 'calibrator.json',
    'eval_report.json'     : WORK_DIR / 'eval_report.json',
    'hyperparameters.json' : WORK_DIR / 'hyperparameters.json',
    'training_curves.png'  : WORK_DIR / 'training_curves.png',
    'roc_pr_curves.png'    : WORK_DIR / 'roc_pr_curves.png',
    'calibration_curve.png': WORK_DIR / 'calibration_curve.png',
    'gradcam_tb_cases.png' : WORK_DIR / 'gradcam_tb_cases.png',
}

print('Output artefacts:')
for name, path in artefacts.items():
    if path.exists():
        size = path.stat().st_size
        print(f'  ✓  {name:<28} {size/1e3:>8.1f} KB  →  {path}')
    else:
        print(f'  ✗  {name:<28} NOT FOUND')

print(f'\nAll output in: {WORK_DIR}')

---
## 16 · Next steps

| Step | Action |
|---|---|
| Multi-center external validation | Re-run eval on Montgomery with Shenzhen-trained model |
| MoCo-CXR pretraining | Run `training/pretrain.py`, set `pretrained='mocov3_cxr'` |
| Edge export | Use `'edge_efficientnet'` preset + `quantize=True` |
| Per-site calibration | `scripts/deploy.py calibrate` on 50-200 local CXRs |
| WHO FIND evaluation | Submit for independent TPP validation |
